## Сессия 2. Фильтрация и выборка данных

In [2]:
import pandas as pd

orders = pd.read_csv('data/orders.csv', sep = ';')

Подготовка данных: Приведение типов данных и исключение незавершенных заказов с предварительной фиксацией

In [7]:

orders['is_delivered'] = orders['order_delivered_customer_date'].notna()
print(orders['is_delivered'].value_counts())
print(f'Исключено из датасета: {orders['is_delivered'].value_counts(normalize = True)[False] * 100.:2f}%')
cleaned_orders = orders[orders['is_delivered']].copy().dropna()
columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in columns:
    cleaned_orders[col] = pd.to_datetime(cleaned_orders[col])
print(cleaned_orders.info())

is_delivered
True     96476
False     2965
Name: count, dtype: int64
Исключено из датасета: 2.981668%
<class 'pandas.core.frame.DataFrame'>
Index: 96461 entries, 0 to 99440
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96461 non-null  object        
 1   customer_id                    96461 non-null  object        
 2   order_status                   96461 non-null  object        
 3   order_purchase_timestamp       96461 non-null  datetime64[ns]
 4   order_approved_at              96461 non-null  datetime64[ns]
 5   order_delivered_carrier_date   96461 non-null  datetime64[ns]
 6   order_delivered_customer_date  96461 non-null  datetime64[ns]
 7   order_estimated_delivery_date  96461 non-null  datetime64[ns]
 8   is_delivered                   96461 non-null  bool          
dtypes: bool(1), datetime64[ns](5), object(3)
memory usage

Подсчет количества заказов, совершенных в 2018 году 

In [ ]:

print(f'количество заказов, совершенных в 2018: {len(cleaned_orders[cleaned_orders['order_delivered_customer_date'].dt.year == 2018])}')


количество заказов, совершенных в 2018: 55274


Подсчет количества отмененных заказов в 2017-году

In [9]:
orders_new = pd.read_csv('data/orders.csv', sep = ';')
columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in columns:
    orders_new[col] = pd.to_datetime(orders_new[col])
print(f"Количество отмененных заказов в 2017: {len(orders_new[(orders_new['order_purchase_timestamp'].dt.year == 2017) & (orders_new['order_status'] == 'canceled')])}")


Количество отмененных заказов в 2017: 265


Выявление проблемных заказов (Заказ считается проблемным, если он был доставлен, но при этом срок доставки привысил 30 дней), подсчет количества проблемных заказов от общего числа заказов, максимальное время доставки заказа, а также выгрузка снапшота проблемных заказов.

In [10]:
delivery_time = (cleaned_orders['order_delivered_customer_date']- cleaned_orders['order_purchase_timestamp']).dt.days
problem_mask = delivery_time > 30
cleaned_orders['problem_order'] = problem_mask
 # Подсчет количества проблемных заказов среди всех доставленыых заказов
problems_percent = len(cleaned_orders[cleaned_orders['problem_order']]) / len(cleaned_orders) * 100
print(f'Процент проблемных заказов: {problems_percent:.2f}%')
print(f'Максимальное время доставки: {delivery_time.max()}')

problem_df = cleaned_orders[cleaned_orders['problem_order']]
problem_df.to_csv('data/problem_orders.csv', index = False, encoding = 'utf-8', sep = ';')

Процент проблемных заказов: 4.27%
Максимальное время доставки: 209


### Ответы на контрольные вопросы: 
1. В чем разница между оператором and и & при фильтрации DataFrame? Почему and выдает ошибку?

     And не используется в pandas так как предназначен для сравнения одиночных элементов, в то время как & работает со всем элементом series поэлементно прогоняя объект через условие.
2. Как отфильтровать строки, где значение колонки НЕ равно определенному (аналог SQL !=)?

    Аналог является метод .ne(), но можно и использовать обычное !=.
3. Почему при фильтрации по дате лучше использовать преобразованный тип datetime, а не строку?

    Потому что доступны различные векторные вычисления + невозможно строку сравнить с числом или отнять из одной даты другую, чтобы получить конкретное количество дней. 

#### Основные инсайты, полученные в ходе анализа: 
- В целом, недоставленные заказы составляют 2.98%, что является норомой для e-commerce платформы и не требует особого внимания
- У платформы существуют проблемы с логистикой, так как 4.27% заказов были доставлены покупателю со значительной задержкой (> 30 дней), стоит обратить внимание на логистические маршруты и выстроить обновленную стратегию работы с компаниями-перевозчиками
- Максимальное время доставки 209 дней, это значительная аномалия, которая требует детального разбора причин
- Необходимо выяснить причины отмены заказов в 2017 и в остальных годах